# API CALLS

## libraries

In [1]:
import requests
import pandas as pd
from datetime import date, timedelta
from tqdm import tqdm

c:\Users\prade\miniconda3\envs\dev_env\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## get data

In [8]:
def get_lamp_data(start_date, end_date):
    dfs = []
    current = start_date
    
    while current <= end_date:
        date_str = current.strftime("%Y-%m-%d")
        url = f"https://performancedata.mbta.com/lamp/subway-on-time-performance-v1/{date_str}-subway-on-time-performance-v1.parquet"
        
        try:
            df = pd.read_parquet(url)
            df["service_date"] = date_str
            df = df[
                    ~df['travel_time_seconds'].isna() &
                    ~df['dwell_time_seconds'].isna()]
            dfs.append(df)
            print(f"✓ Loaded {date_str} — {len(df)} rows")
        except Exception as e:
            print(f"✗ Skipping {date_str}: {e}")
        
        current += timedelta(days=1)
    
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


In [9]:
# data from oct 2025
df_oct25 = get_lamp_data(
    start_date = date(2025, 10, 1),
    end_date   = date(2025, 10, 31)
)

print(df_oct25.shape)

# data from oct 2025
df_oct26 = get_lamp_data(
    start_date = date(2025, 10, 1),
    end_date   = date(2025, 10, 31)
)

print(df_oct26.shape)

df = pd.concat([df_oct25, df_oct26], ignore_index=True).reset_index(drop=True)

✓ Loaded 2025-10-01 — 41643 rows
✓ Loaded 2025-10-02 — 40046 rows
✓ Loaded 2025-10-03 — 40704 rows
✓ Loaded 2025-10-04 — 30517 rows
✓ Loaded 2025-10-05 — 28400 rows
✓ Loaded 2025-10-06 — 41411 rows
✓ Loaded 2025-10-07 — 41771 rows
✓ Loaded 2025-10-08 — 42511 rows
✓ Loaded 2025-10-09 — 41083 rows
✓ Loaded 2025-10-10 — 42220 rows
✓ Loaded 2025-10-11 — 28800 rows
✓ Loaded 2025-10-12 — 27841 rows
✓ Loaded 2025-10-13 — 28719 rows
✓ Loaded 2025-10-14 — 42070 rows
✓ Loaded 2025-10-15 — 41222 rows
✓ Loaded 2025-10-16 — 41304 rows
✓ Loaded 2025-10-17 — 43297 rows
✓ Loaded 2025-10-18 — 33123 rows
✓ Loaded 2025-10-19 — 31463 rows
✓ Loaded 2025-10-20 — 41356 rows
✓ Loaded 2025-10-21 — 42533 rows
✓ Loaded 2025-10-22 — 41327 rows
✓ Loaded 2025-10-23 — 41444 rows
✓ Loaded 2025-10-24 — 40777 rows
✓ Loaded 2025-10-25 — 29724 rows
✓ Loaded 2025-10-26 — 27701 rows
✓ Loaded 2025-10-27 — 37613 rows
✓ Loaded 2025-10-28 — 38129 rows
✓ Loaded 2025-10-29 — 35322 rows
✓ Loaded 2025-10-30 — 37443 rows
✓ Loaded 2

In [26]:
a = sorted(list(df.columns))
a

['branch_route_id',
 'direction',
 'direction_destination',
 'direction_id',
 'dwell_time_seconds',
 'headway_branch_seconds',
 'headway_trunk_seconds',
 'move_timestamp',
 'parent_station',
 'route_id',
 'scheduled_arrival_time',
 'scheduled_departure_time',
 'scheduled_headway_branch',
 'scheduled_headway_trunk',
 'scheduled_travel_time',
 'service_date',
 'start_time',
 'stop_count',
 'stop_id',
 'stop_sequence',
 'stop_timestamp',
 'travel_time_seconds',
 'trip_id',
 'trunk_route_id',
 'vehicle_consist',
 'vehicle_id',
 'vehicle_label']

## breakdown of every column:

**Stop Info**
| Column | Description |
|--------|-------------|
| `stop_sequence` | Order of this stop within the trip (1 = first stop) |
| `stop_id` | Unique ID of the specific stop/platform |
| `parent_station` | Station the stop belongs to (e.g. Park St has multiple platforms) |
| `stop_count` | Total number of stops in the trip |

**Timestamps**
| Column | Description |
|--------|-------------|
| `stop_timestamp` | Actual time train **arrived** at stop (Unix seconds) |
| `move_timestamp` | Actual time train **departed** stop (Unix seconds) |
| `start_time` | Actual time the trip started |
| `service_date` | Operating date (note: trips after midnight still belong to previous service date) |

**Travel & Performance**
| Column | Description |
|--------|-------------|
| `travel_time_seconds` | Actual time taken to travel from previous stop to this stop |
| `dwell_time_seconds` | Actual time spent at this stop (move - stop timestamp) |
| `headway_trunk_seconds` | Actual time since last train on the **main line** |
| `headway_branch_seconds` | Actual time since last train on the **branch** |

**Scheduled (baseline to compare against)**
| Column | Description |
|--------|-------------|
| `scheduled_arrival_time` | Scheduled arrival at this stop (Unix seconds) |
| `scheduled_departure_time` | Scheduled departure from this stop (Unix seconds) |
| `scheduled_travel_time` | Scheduled travel time from previous stop |
| `scheduled_headway_trunk` | Scheduled headway on main line |
| `scheduled_headway_branch` | Scheduled headway on branch |

**Trip & Route Info**
| Column | Description |
|--------|-------------|
| `trip_id` | Unique ID for this specific trip/run |
| `route_id` | Route (e.g. `Red`, `Orange`, `Green-B`) |
| `branch_route_id` | Branch of the route (e.g. `Green-B`, `Green-C`) |
| `trunk_route_id` | Main trunk line shared by branches (e.g. `Green`) |
| `direction_id` | `0` = outbound, `1` = inbound |
| `direction` | Text direction e.g. `"Southbound"` |
| `direction_destination` | Final destination e.g. `"Alewife"` |

**Vehicle Info**
| Column | Description |
|--------|-------------|
| `vehicle_id` | Unique ID of the physical train/vehicle |
| `vehicle_label` | Human-readable train number |
| `vehicle_consist` | Which cars make up the train |

In [10]:
# convert timestamp
# Convert all timestamp columns at once
timestamp_cols = [
    "stop_timestamp",
    "move_timestamp", 
    "scheduled_arrival_time",
    "scheduled_departure_time",
    "start_time"
]

for col in timestamp_cols:
    df[col] = pd.to_datetime(df[col], unit="s", utc=True).dt.tz_convert("America/New_York")

In [28]:
df

,stop_sequence,stop_id,parent_station,move_timestamp,stop_timestamp,travel_time_seconds,dwell_time_seconds,headway_trunk_seconds,headway_branch_seconds,service_date,...,trip_id,vehicle_label,vehicle_consist,direction,direction_destination,scheduled_arrival_time,scheduled_departure_time,scheduled_travel_time,scheduled_headway_branch,scheduled_headway_trunk
0,320,70162,place-woodl,2025-10-01 04:53:10-04:00,2025-10-01 04:54:14-04:00,64.0,26.0,172.0,172.0,2025-10-01,...,ADDED-1583148116,3834-3833,3834|3833,East,Union Square,1970-01-01 13:19:00-05:00,1970-01-01 13:19:00-05:00,60.0,480.0,480.0
1,330,70164,place-waban,2025-10-01 04:54:40-04:00,2025-10-01 04:56:07-04:00,87.0,26.0,149.0,149.0,2025-10-01,...,ADDED-1583148116,3834-3833,3834|3833,East,Union Square,1970-01-01 13:22:00-05:00,1970-01-01 13:22:00-05:00,180.0,480.0,480.0
2,670,70513,place-esomr,2025-10-01 04:55:13-04:00,2025-10-01 04:56:40-04:00,87.0,27.0,572.0,572.0,2025-10-01,...,ADDED-1583148124,3674-3814,3674|3814,East,Medford/Tufts,1970-01-01 05:12:00-05:00,1970-01-01 05:12:00-05:00,240.0,420.0,420.0
3,340,70166,place-eliot,2025-10-01 04:56:33-04:00,2025-10-01 04:57:56-04:00,83.0,27.0,125.0,125.0,2025-10-01,...,ADDED-1583148116,3834-3833,3834|3833,East,Union Square,1970-01-01 13:24:00-05:00,1970-01-01 13:24:00-05:00,120.0,480.0,480.0
4,680,70505,place-gilmn,2025-10-01 04:57:07-04:00,2025-10-01 04:58:19-04:00,72.0,25.0,570.0,570.0,2025-10-01,...,ADDED-1583148124,3674-3814,3674|3814,East,Medford/Tufts,1970-01-01 05:14:00-05:00,1970-01-01 05:14:00-05:00,120.0,420.0,420.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1600411,600,70200,place-pktrm,2025-11-01 02:50:13-04:00,2025-11-01 02:51:15-04:00,62.0,32.0,1837.0,1837.0,2025-10-31,...,ADDED-1583239532,3857-3674,3857|3674,East,Medford/Tufts,1970-01-01 18:59:00-05:00,1970-01-01 18:59:00-05:00,60.0,540.0,60.0
1600412,610,70201,place-gover,2025-11-01 02:51:47-04:00,2025-11-01 02:52:43-04:00,56.0,17.0,1842.0,1842.0,2025-10-31,...,ADDED-1583239532,3857-3674,3857|3674,East,Medford/Tufts,1970-01-01 19:01:00-05:00,1970-01-01 19:01:00-05:00,120.0,540.0,60.0
1600413,620,70203,place-haecl,2025-11-01 02:53:00-04:00,2025-11-01 02:54:00-04:00,60.0,30.0,1688.0,1757.0,2025-10-31,...,ADDED-1583239532,3857-3674,3857|3674,East,Medford/Tufts,1970-01-01 19:03:00-05:00,1970-01-01 19:03:00-05:00,120.0,540.0,60.0
1600414,630,70205,place-north,2025-11-01 02:54:30-04:00,2025-11-01 02:55:43-04:00,73.0,24.0,1612.0,1675.0,2025-10-31,...,ADDED-1583239532,3857-3674,3857|3674,East,Medford/Tufts,1970-01-01 19:05:00-05:00,1970-01-01 19:05:00-05:00,120.0,540.0,60.0


## get stop from stop id

In [5]:
stops = pd.read_parquet("https://performancedata.mbta.com/lamp/tableau/rail/LAMP_static_stops.parquet")
stop_names = stops.set_index('stop_id')['stop_name'].to_dict()

In [17]:
stops.to_csv(r'../data/stop_name.csv', index=False)

## save data frame

In [11]:
# set stop name
df['stop_id'] = df['stop_id'].astype(str)
df['stop_name'] = df['stop_id'].apply(lambda x: stop_names[x])

In [12]:
df.to_csv(r'..\data\processed_oct_Data.csv',
          index=False)

In [7]:
df.stop_name.unique()

array(['Lechmere', 'Boston College', 'Union Square', 'Riverside',
       'Ashmont', 'Cleveland Circle', 'East Somerville', 'Orient Heights',
       'Braintree', 'Gilman Square', 'Magoun Square', 'Woodland',
       'Ball Square', 'Waban', 'Medford/Tufts', 'Beaconsfield',
       'Brookline Hills', 'Eliot', 'Quincy Adams',
       'Science Park/West End', 'Brookline Village', 'Newton Highlands',
       'North Station', 'Longwood', 'Wonderland', 'Newton Centre',
       'Fenway', 'Englewood Avenue', 'Capen Street', 'South Street',
       'Dean Road', 'Valley Road', 'Chestnut Hill', 'Kenmore',
       'Haymarket', 'Tappan Street', 'Central Avenue', 'Oak Grove',
       'Washington Square', 'Chestnut Hill Avenue', 'Government Center',
       'Milton', 'Fairbanks Street', 'Hynes Convention Center',
       'Chiswick Road', 'Reservoir', 'Brandon Hall', 'Butler',
       'Summit Avenue', 'Sutherland Road', 'Coolidge Corner',
       'Cedar Grove', 'Park Street', 'Copley', 'Washington Street',
       '